# K-Nearest Neighbors: Classifying Wine Cultivars

**Family:** Supervised learning | **Level:** Beginner | **Dataset:** `wine` | **Runs on:** CPU, about 1 minute(s)

Classify by voting among the closest examples and see why feature scaling decides everything.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- How the nearest neighbour rule works
- Why unscaled features break distance based models
- How k trades bias against variance
- Why high dimensions hurt KNN

**Prerequisites:**

- Distance and vectors
- Confusion matrix basics

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Wine authenticity and quality control use chemical analysis to tell the grape cultivar.

**The problem.** Identify the cultivar of an Italian wine from 13 chemical measurements.

**Why it matters.** Labelling fraud and blending errors are detected from chemistry without a tasting panel.

**Why machine learning.** Cultivars form groups in chemical space.

**Why this algorithm.** KNN makes no model assumption, which makes it a good first check of whether classes are separable by distance.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### Nearest neighbor pattern classification

- **Type:** foundational paper
- **Authors:** Cover, T. and Hart, P.
- **Year and venue:** 1967, IEEE Transactions on Information Theory 13(1):21-27
- **DOI or URL:** https://doi.org/10.1109/TIT.1967.1053964
- **Verification:** Crossref 2026-11-17
- **Research objective:** Bound the error of the nearest neighbour rule.
- **Methodology:** Asymptotic analysis of the 1-NN rule against the Bayes error.
- **Dataset:** Theoretical paper, no empirical dataset.
- **Main findings:** Asymptotically the 1-NN error is at most twice the Bayes error.
- **Limitations:** Asymptotic result. Says little about small samples or high dimensions.

### The Elements of Statistical Learning (2nd ed.)

- **Type:** foundational paper
- **Authors:** Hastie, T., Tibshirani, R. and Friedman, J.
- **Year and venue:** 2009, Springer
- **DOI or URL:** https://doi.org/10.1007/978-0-387-84858-7
- **Verification:** not verified in build environment
- **Research objective:** Textbook on statistical learning.
- **Methodology:** Textbook.
- **Dataset:** Not applicable.
- **Main findings:** Reference text.
- **Limitations:** Textbook.

**Relationship to this notebook.** Cover and Hart prove the asymptotic error bound of the nearest neighbour rule. Our small sample is far from that asymptotic regime. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Rule.** For a query $x$ find the $k$ training points with smallest distance $d(x, x_i)$ and predict the majority label:

$$\hat y(x) = \arg\max_c \sum_{i \in \mathcal{N}_k(x)} \mathbb{1}[y_i = c]$$

**Distance.** Euclidean $d(x,x') = \lVert x - x' \rVert_2$ or Manhattan $\lVert x - x' \rVert_1$.

**Asymptotic bound.** For $k=1$ and $n \to \infty$, the error is at most $2 R^*(1-R^*)$ with $R^*$ the Bayes error, so at most twice the Bayes error (Cover and Hart, 1967).

**Assumptions.** Nearby points share labels, all features are on comparable scales.

**Limitations.** Prediction cost grows with data size. Distances lose meaning in many dimensions. No training, so no interpretable parameters.

## 5. Dataset Introduction

**Wine**, UCI, licence CC BY 4.0, DOI 10.24432/C5PC7J. 178 rows, 13 chemical measurements (alcohol, malic acid, ash, magnesium, phenols, flavanoids, colour intensity, proline and others). Target `cultivar`, three classes. Limits: one region, tiny sample.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "k_nearest_neighbors"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("wine")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. Compare the ranges of `proline` (hundreds to over a thousand) and `hue` (below 2).

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.


In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=True, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

The model is the training set. A new point is labelled by its neighbours, so the geometry of the feature space is everything.

### 8.2 Python implementation from scratch

KNN is short enough to write in a few NumPy lines.

In [ ]:
ytr, yva = y_train.values, y_val.values

def knn_predict(X_fit, y_fit, X_query, k=5):
    dist = np.sqrt(((X_query[:, None, :] - X_fit[None, :, :]) ** 2).sum(axis=2))
    nearest = np.argsort(dist, axis=1)[:, :k]
    votes = y_fit[nearest]
    return np.array([np.bincount(row).argmax() for row in votes])

mine = knn_predict(Xtr, ytr, Xva, k=5)
ref = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr).predict(Xva)
print("agreement with sklearn:", np.mean(mine == ref), "| validation accuracy:", np.mean(mine == yva))

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `n_neighbors` | Number of voters $k$. |
| `weights` | `uniform` or `distance` weighting. |
| `metric` / `p` | Distance type. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "F1"
LOWER_IS_BETTER = False
SCORING = "f1_macro"
SWEEP_PARAM = "model__n_neighbors"
SWEEP_VALUES = [1, 3, 5, 7, 9, 11, 15, 21]

def build_pipeline():
    return make_model_pipeline(X_train, KNeighborsClassifier(), scale=True, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Classification metrics: accuracy, precision, recall, F1, ROC-AUC and PR-AUC, plus the confusion matrix. With imbalanced classes accuracy alone misleads, so the model is chosen on F1 or PR-AUC as stated in the training cell.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "KNeighborsClassifier": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="KNeighborsClassifier", dataset="wine",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
labels = bundle.target_names or None
fig = viz.plot_confusion(y_test, pred, labels=labels)
save_figure(fig, NOTEBOOK_ID, "confusion"); plt.show()
if y_test.nunique() == 2:
    fig = viz.plot_roc_pr(final_model, X_test, y_test)
    save_figure(fig, NOTEBOOK_ID, "roc_pr"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()
# decision regions in two PCA axes
pca_view = PCA(n_components=2, random_state=SEED).fit(Xtr)
model_for_plot = clone(final_model.named_steps["model"])
fig = viz.plot_decision_boundary(lambda: clone(model_for_plot), pca_view.transform(Xtr), y_train.values,
                                 names=["PCA 1", "PCA 2"])
save_figure(fig, NOTEBOOK_ID, "decision_regions"); plt.show()

## 12. Interpretation

Permutation importance shows which inputs the model relies on. The confusion matrix shows which classes are mixed up, and the list of misclassified rows is the starting point for error analysis. A model is only as good as the cases it fails on.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring=SCORING, n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
fig = viz.plot_importance(importance.index, importance.values, title="Permutation importance on the test set",
                          top=12, xlabel="Drop in " + SCORING + " when shuffled")
save_figure(fig, NOTEBOOK_ID, "importance"); plt.show()
print("Top features:", ", ".join(importance.index[:5]))
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Scaled versus unscaled features

Repeat the cross validation with and without standardisation.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_f1(model, scoring="f1"):
    scores = cross_val_score(model, X_trainval, y_trainval, cv=5, scoring=scoring)
    return scores.mean(), scores.std(ddof=1)
for scaled in [True, False]:
    mean, std = cv_f1(make_model_pipeline(X_trainval, KNeighborsClassifier(n_neighbors=5), scale=scaled), scoring="f1_macro")
    print(f"scaled={scaled}: CV macro F1 {mean:.3f} +- {std:.3f}")

### Experiment 2: Curse of dimensionality

Append pure noise features and watch the score fall, because noise dominates the distance.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_f1(model, scoring="f1"):
    scores = cross_val_score(model, X_trainval, y_trainval, cv=5, scoring=scoring)
    return scores.mean(), scores.std(ddof=1)
rng = np.random.default_rng(SEED)
for extra in [0, 10, 50, 200]:
    X_noise = X_trainval.copy()
    for j in range(extra):
        X_noise[f"noise{j}"] = rng.normal(size=len(X_noise))
    mean, std = cv_f1(make_model_pipeline(X_noise, KNeighborsClassifier(n_neighbors=5)), scoring="f1_macro") if False else (None, None)
    from sklearn.model_selection import cross_val_score
    s = cross_val_score(make_model_pipeline(X_noise, KNeighborsClassifier(n_neighbors=5)), X_noise, y_trainval, cv=5, scoring="f1_macro")
    print(f"{extra} noise features: CV macro F1 {s.mean():.3f} +- {s.std(ddof=1):.3f}")

### Experiment 3: Weights and metric

Compare uniform and distance weighting, Euclidean and Manhattan distance.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_f1(model, scoring="f1"):
    scores = cross_val_score(model, X_trainval, y_trainval, cv=5, scoring=scoring)
    return scores.mean(), scores.std(ddof=1)
for weights in ["uniform", "distance"]:
    for p in [1, 2]:
        mean, std = cv_f1(make_model_pipeline(X_trainval, KNeighborsClassifier(n_neighbors=7, weights=weights, p=p)), scoring="f1_macro")
        print(f"weights={weights}, p={p}: CV macro F1 {mean:.3f} +- {std:.3f}")

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Change the metric used for model choice and see whether the chosen value changes.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Cover and Hart bound the 1-NN error asymptotically. Our k sweep shows the practical side: with 178 rows the best k is found empirically and results vary between splits, which the cross validation spread makes visible.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Use feature weighting or metric learning, approximate neighbour search for large data, and repeated cross validation for small samples.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- Always scale features for KNN.
- k controls smoothness.
- Noise features hurt distance based models.

**Use cases.** Small data baselines, recommendation neighbours, anomaly scoring.

**Limitations.** Slow at prediction time, weak in high dimensions.

**Recommended next notebook.** Decision Tree.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.